<a href="https://colab.research.google.com/github/JayasreeMeda/DataManagement/blob/main/ps2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ps2 - manipulating data + merging
**Jayasree Meda**

**Research question (abstract).** ps1 showed how lopsided world population is: a handful of countries hold most of humanity. But population tells you HOW MANY people live in a country, not WHERE, packed into cities or spread across countryside. This ps2 merges the ps1 population panel with the World Bank urban-population share, then asks: **are the world's biggest countries also the most urbanized, or do the giants still have most of their people living rural?**

**Unit of analysis.** Country (merged frame). One sidebar keeps NHANES person-level data from ps1 for a US-inside view.

**Data**
1. HTML, Wikipedia countries by population, `https://en.wikipedia.org/wiki/List_of_countries_by_population_(United_Nations)` (ps1)
2. JSON, World Bank population 2022, `https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL?format=json&date=2022&per_page=400` (ps1)
3. SAS xpt, NHANES 2021-2023 Demographics, `https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DEMO_L.xpt` (ps1)
4. JSON (new), World Bank Urban population (% of total), 2022 (`SP.URB.TOTL.IN.ZS`), `https://api.worldbank.org/v2/country/all/indicator/SP.URB.TOTL.IN.ZS?format=json&date=2022&per_page=400`


In [1]:
import io, json, urllib.request
import numpy as np
import pandas as pd

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 14)

UA = {"User-Agent": "Mozilla/5.0 (ps2 educational)"}


## 1. Rebuild the ps1 datasets

In [2]:
# 1a. Wikipedia countries by population (HTML)
URL_POP = "https://en.wikipedia.org/wiki/List_of_countries_by_population_(United_Nations)"
html = urllib.request.urlopen(urllib.request.Request(URL_POP, headers=UA)).read().decode()
pop_wiki = pd.read_html(io.StringIO(html))[0]
pop_wiki = pop_wiki.rename(columns={
    "Country or territory": "country",
    "Population (1 July 2023)": "population",
    "Change (%)": "change_pct",
    "UN continental region[1]": "region",
})[["country", "population", "change_pct", "region"]]
pop_wiki = pop_wiki.dropna(subset=["country", "population"])
pop_wiki = pop_wiki[pop_wiki["country"] != "World"].reset_index(drop=True)
pop_wiki["change_pct"] = pd.to_numeric(
    pop_wiki["change_pct"].str.replace("%","",regex=False).str.replace("−","-",regex=False),
    errors="coerce")
pop_wiki.shape, pop_wiki.head()


((237, 4),
          country    population  change_pct    region
 0          India  1.438070e+09        0.89      Asia
 1       China[a]  1.422585e+09       -0.18      Asia
 2  United States  3.434773e+08        0.57  Americas
 3      Indonesia  2.811901e+08        0.85      Asia
 4       Pakistan  2.475045e+08        1.56      Asia)

In [3]:
# 1b. World Bank population 2022 (JSON)
URL_WB_POP = ("https://api.worldbank.org/v2/country/all/indicator/"
              "SP.POP.TOTL?format=json&date=2022&per_page=400")
payload = json.load(urllib.request.urlopen(URL_WB_POP))
wb_pop = pd.json_normalize(payload[1])
wb_pop = (wb_pop[["country.value","countryiso3code","value"]]
          .rename(columns={"country.value":"country","countryiso3code":"iso3","value":"population"}))
wb_pop = wb_pop.dropna(subset=["population","iso3"])
wb_pop = wb_pop[wb_pop["iso3"].str.len()==3].reset_index(drop=True)
wb_pop.shape, wb_pop.head()


((260, 3),
                           country iso3   population
 0     Africa Eastern and Southern  AFE  731821393.0
 1      Africa Western and Central  AFW  496366058.0
 2                      Arab World  ARB  471352066.0
 3          Caribbean small states  CSS    4497310.0
 4  Central Europe and the Baltics  CEB  100071871.0)

In [4]:
# 1c. NHANES demographics (SAS xpt)
URL_DEMO = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DEMO_L.xpt"
demo_full = pd.read_sas(URL_DEMO, format="xport")
demo = (demo_full[["SEQN","RIAGENDR","RIDAGEYR","RIDRETH3","INDFMPIR"]]
        .rename(columns={"SEQN":"id","RIAGENDR":"gender","RIDAGEYR":"age",
                         "RIDRETH3":"race","INDFMPIR":"poverty_ratio"}))
demo["gender"] = demo["gender"].map({1:"Male", 2:"Female"})
demo["race"]   = demo["race"].map({1:"Mex-American", 2:"Other Hispanic",
                                   3:"NH-White",     4:"NH-Black",
                                   6:"NH-Asian",     7:"Other"})
demo.shape, demo.head()


((11933, 5),
          id  gender   age            race  poverty_ratio
 0  130378.0    Male  43.0        NH-Asian           5.00
 1  130379.0    Male  66.0        NH-White           5.00
 2  130380.0  Female  44.0  Other Hispanic           1.41
 3  130381.0  Female   5.0           Other           1.53
 4  130382.0    Male   2.0        NH-White           3.60)

## 2. New dataset, World Bank urban population share

In [5]:
URL_URB = ("https://api.worldbank.org/v2/country/all/indicator/"
           "SP.URB.TOTL.IN.ZS?format=json&date=2022&per_page=400")
payload = json.load(urllib.request.urlopen(URL_URB))
urb = pd.json_normalize(payload[1])
urb = (urb[["country.value","countryiso3code","value"]]
       .rename(columns={"country.value":"country","countryiso3code":"iso3","value":"urban_pct"}))
urb = urb.dropna(subset=["urban_pct","iso3"])
urb = urb[urb["iso3"].str.len()==3].reset_index(drop=True)
urb.shape, urb.head()


((260, 3),
                           country iso3  urban_pct
 0     Africa Eastern and Southern  AFE  37.360578
 1      Africa Western and Central  AFW  51.700103
 2                      Arab World  ARB  59.333933
 3          Caribbean small states  CSS  50.131445
 4  Central Europe and the Baltics  CEB  61.593524)

## 3. Merge, population and urban share

In [6]:
# Filter both frames to real sovereign countries.
# WB's country metadata endpoint tags aggregates with region.id == "NA".
META = json.load(urllib.request.urlopen("https://api.worldbank.org/v2/country?format=json&per_page=400"))[1]
REAL_ISO3 = {c["id"] for c in META if c.get("region", {}).get("id") != "NA"}
len(REAL_ISO3)


217

In [7]:
wb_pop_c = wb_pop[wb_pop["iso3"].isin(REAL_ISO3)].copy()
urb_c    = urb[urb["iso3"].isin(REAL_ISO3)].copy()

merged = wb_pop_c.merge(urb_c[["iso3","urban_pct"]], on="iso3", how="inner")
merged["urban_pop"] = (merged["population"] * merged["urban_pct"] / 100).round(0)
merged = merged[["country","iso3","population","urban_pct","urban_pop"]].reset_index(drop=True)
merged.shape, merged.head()


((217, 5),
           country iso3  population  urban_pct   urban_pop
 0     Afghanistan  AFG  40578842.0  25.393925  10304561.0
 1         Albania  ALB   2451636.0  57.864348   1418623.0
 2         Algeria  DZA  45477389.0  74.179586  33734939.0
 3  American Samoa  ASM     48342.0  81.794798     39541.0
 4         Andorra  AND     79705.0  88.754877     70742.0)

In [8]:
# Merge validation
print("row count:", len(merged))
print("countries lost from WB pop:", len(wb_pop) - len(merged))
merged.isna().sum()


row count: 217
countries lost from WB pop: 43


,0
country,0
iso3,0
population,0
urban_pct,0
urban_pop,0


In [9]:
# Sanity spot-check
merged[merged["country"].isin(["India","China","United States","Singapore","Ethiopia"])]


,country,iso3,population,urban_pct,urban_pop
41,China,CHN,1.412175e+09,65.217636,920987153.0
64,Ethiopia,ETH,1.253843e+08,22.736769,28508335.0
89,India,IND,1.425423e+09,34.755567,495413925.0
171,Singapore,SGP,5.637022e+06,100.000000,5637022.0
206,United States,USA,3.339963e+08,80.032498,267305585.0


## 4. Manipulations (2x each, all different from ps1)

### 4a. map / recode

In [10]:
# Recode 1: urban_pct bucket -> settlement_type
merged["settlement_type"] = pd.cut(merged["urban_pct"],
                                    bins=[0, 30, 60, 100],
                                    labels=["Rural","Mixed","Urban"])
merged["settlement_type"].value_counts()


,count
settlement_type,
Urban,122
Mixed,71
Rural,24


In [11]:
# Recode 2: NHANES age -> life stage
demo["life_stage"] = pd.cut(demo["age"],
                             bins=[-0.1, 12, 17, 64, 200],
                             labels=["Child","Teen","Adult","Senior"])
demo["life_stage"].value_counts()


,count
life_stage,
Adult,5567
Child,2773
Senior,2586
Teen,1007


### 4b. replace on condition

In [12]:
# Replace 1: cap urban_pct at 99.9 for city-states so it does not sit exactly at 100
merged.loc[merged["urban_pct"] >= 100, "urban_pct"] = 99.9
merged["urban_pct"].agg(["max","min"])


,urban_pct
max,99.900000
min,14.692307


In [13]:
# Replace 2: NHANES top-codes age at 80. Make that explicit for anyone reusing the frame.
demo.loc[demo["age"] > 80, "age"] = 80
demo["age"].describe().round(1)


,age
count,11933.0
mean,38.3
std,25.6
min,0.0
25%,13.0
50%,37.0
75%,62.0
max,80.0


### 4c. subset / slice

In [14]:
# Subset 1: big countries that are still mostly rural (pop > 50M AND urban < 50%)
big_rural = merged.query("population > 50_000_000 and urban_pct < 50").sort_values("population", ascending=False)
big_rural


,country,iso3,population,urban_pct,urban_pop,settlement_type
89,India,IND,1.425423e+09,34.755567,495413925.0,Mixed
149,Pakistan,PAK,2.437007e+08,38.374044,93517801.0,Mixed
15,Bangladesh,BGD,1.693849e+08,31.673735,53650523.0,Mixed
64,Ethiopia,ETH,1.253843e+08,22.736769,28508335.0,Rural
58,"Egypt, Arab Rep.",EGY,1.126182e+08,42.632513,48011990.0,Mixed
44,"Congo, Dem. Rep.",COD,1.023970e+08,44.003288,45058032.0,Mixed
211,Viet Nam,VNM,9.968066e+07,37.547852,37427944.0,Mixed
191,Tanzania,TZA,6.471182e+07,34.823474,22534904.0,Mixed
101,Kenya,KEN,5.425246e+07,31.430782,17051973.0,Mixed
135,Myanmar,MMR,5.375679e+07,30.311096,16294271.0,Mixed


In [15]:
# Subset 2: NHANES race isin two categories
demo_sel = demo[demo["race"].isin(["NH-Asian","NH-Black"])]
demo_sel["race"].value_counts()


,count
race,
NH-Black,1597
NH-Asian,681


### 4d. groupby / agg

In [16]:
# Group 1: totals by settlement type. How many people live rural vs urban worldwide?
merged.groupby("settlement_type", observed=True).agg(
    n=("country","count"),
    total_pop=("population","sum"),
    total_urban_pop=("urban_pop","sum"),
).round(0)


,n,total_pop,total_urban_pop
settlement_type,,,
Rural,24,3.859281e+08,9.077064e+07
Mixed,71,3.415034e+09,1.383419e+09
Urban,122,4.164402e+09,3.066762e+09


In [17]:
# Group 2: NHANES life-stage x gender mean poverty ratio
demo.groupby(["life_stage","gender"], observed=True)["poverty_ratio"].mean().round(2).unstack()


gender,Female,Male
life_stage,,
Child,2.32,2.38
Teen,2.35,2.38
Adult,2.72,2.97
Senior,2.85,3.08


## 5. Interpretation

_Interpretation._ The world splits cleanly into three settlement tiers. The Urban tier holds the majority of humanity today, but a small number of very large "Mixed" countries (India, Pakistan, Ethiopia, Bangladesh, Egypt) still have half their people in the countryside. Those five together carry more rural population than the entire Rural tier does. The NHANES sidebar shows the age/poverty structure inside one fully-urbanized country, where the child and senior life-stages sit at very different mean poverty ratios.